# Tutorial: Custom Iterators and Predicates

This notebook shows how to extend nkDSL with one custom iterator and one custom predicate, then use both as normal fluent methods in a symbolic operator.


## What you will build

By the end, you will be able to:

- define and register a `for_each_even_site(...)` iterator clause
- define and register an `at_least_occupancy(...)` predicate clause
- compose them with `emit(...)` and `where(...)` in one operator
- inspect the resulting connected states and matrix elements


## 1) Setup


In [2]:
import jax.numpy as jnp
import netket as nk
import nkdsl

L = 6
hi = nk.hilbert.Fock(n_max=3, N=L)


## 2) Create a custom iterator clause

This iterator will visit only even lattice sites.

Important pattern: subclass `nkdsl.AbstractIteratorClause`, implement `build_iterator(...)`, and register with `replace=True` while iterating in notebooks.


In [3]:
class ForEachEvenSite(nkdsl.AbstractIteratorClause):
    clause_name = "for_each_even_site"

    def build_iterator(self, hilbert, label: str = "i"):
        n = int(hilbert.size)
        rows = tuple((k,) for k in range(n) if k % 2 == 0)
        if not rows:
            raise ValueError("No even sites available for this Hilbert space.")
        return (str(label),), rows

nkdsl.register_iterator_clause(ForEachEvenSite, replace=True)


__main__.ForEachEvenSite

In [4]:
'for_each_even_site' in nkdsl.available_iterator_clause_names()


True

## 3) Create a custom predicate clause

This predicate keeps only sites whose occupancy is at least a configurable cutoff.


In [5]:
class AtLeastOccupancy(nkdsl.AbstractPredicateClause):
    clause_name = "at_least_occupancy"

    def build_predicate(self, ctx, label: str = "i", cutoff: int = 1):
        return ctx.site(label).value >= int(cutoff)

nkdsl.register_predicate_clause(AtLeastOccupancy, replace=True)


__main__.AtLeastOccupancy

In [6]:
'at_least_occupancy' in nkdsl.available_predicate_clause_names()


True

## 4) Use both clauses in one symbolic operator

Now both custom methods are available directly on `SymbolicDiscreteJaxOperator`.

- iterator: picks even sites
- predicate: keeps those with occupancy >= 1
- emission: diagonal (`identity`) with matrix element equal to the selected site value


In [7]:
op = (
    nkdsl.SymbolicDiscreteJaxOperator(hi, "custom-even-filter")
    .for_each_even_site("i")
    .at_least_occupancy("i", cutoff=1)
    .emit(nkdsl.identity(), matrix_element=nkdsl.site("i").value)
    .build()
    .compile()
)

op.max_conn_size


3

## 5) Inspect connectivity on one configuration

Here, only even-site visits that pass the predicate contribute nonzero matrix elements.


In [8]:
x = jnp.asarray([0, 2, 1, 3, 2, 0], dtype=jnp.int32)
xp, mels = op.get_conn_padded(x)

print('input state:', x)
print('connected states shape:', xp.shape)
print('matrix elements:', mels)


input state: [0 2 1 3 2 0]
connected states shape: (3, 6)
matrix elements: [0. 1. 2.]


## 6) Compose with `where(...)`

Custom predicate clauses compose with built-in `where(...)` via logical AND.


In [9]:
op_strict = (
    nkdsl.SymbolicDiscreteJaxOperator(hi, "custom-even-filter-strict")
    .for_each_even_site("i")
    .at_least_occupancy("i", cutoff=1)
    .where(nkdsl.site("i").value < 2)
    .emit(nkdsl.identity(), matrix_element=nkdsl.site("i").value)
    .build()
    .compile()
)

_xp2, mels2 = op_strict.get_conn_padded(x)
print('matrix elements after extra where:', mels2)


matrix elements after extra where: [0. 1. 0.]


## Recap and next steps

You now have the full extension workflow:

1. Subclass `AbstractIteratorClause` / `AbstractPredicateClause`.
2. Register with `register_iterator_clause(...)` / `register_predicate_clause(...)`.
3. Use the new methods fluently on `SymbolicDiscreteJaxOperator`.

For API-level details and additional patterns, see:

- [Extending DSL Iterators](https://nkdsl.readthedocs.io/en/latest/guides/extending_dsl/iterators.html)
- [Extending DSL Predicates](https://nkdsl.readthedocs.io/en/latest/guides/extending_dsl/predicates.html)

